In [2]:
print("Installing Apache Airflow... (This may take around 30-45 seconds)")
!pip install apache-airflow > /dev/null 2>&1
print("✔ Apache Airflow installed successfully!\n")

Installing Apache Airflow... (This may take around 30-45 seconds)
✔ Apache Airflow installed successfully!



In [9]:
import os
from datetime import datetime
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator

def create_bill_file():
    file_path = "/tmp/electricity.txt"
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    content = "Rahul,210\nPriya,180\nAmit,300\nSneha,150\nKiran,260"
    with open(file_path, "w") as f:
        f.write(content.strip())
    print(f"✔ Created {file_path}")

def calculate_total_units():
    file_path = "/tmp/electricity.txt"
    total_units = 0
    customer_count = 0

    with open(file_path, "r") as f:
        for line in f:
            if line.strip():
                _, units = line.strip().split(",")
                total_units += int(units)
                customer_count += 1

    avg_units = total_units / customer_count
    print(f"Calculated: Customers={customer_count}, Total={total_units}, Avg={avg_units}")
    return customer_count, total_units, avg_units

def generate_bill_summary():
    # Recalculate metrics for execution simplicity
    customer_count, total_units, avg_units = calculate_total_units()
    report_path = "/tmp/bill_summary.txt"

    with open(report_path, "w") as f:
        f.write(f"Customers = {customer_count}\n")
        f.write(f"Total Units = {total_units}\n")
        f.write(f"Average Units = {int(avg_units)}\n")

    print(f"✔ Generated Summary at {report_path}")
    print("\n📋 File Content:")
    with open(report_path, "r") as f: print(f.read())

with DAG(dag_id='exercise_8_electricity', start_date=datetime(2026, 1, 1), schedule=None, catchup=False) as dag:
    t1 = PythonOperator(task_id='create_bill_file', python_callable=create_bill_file)
    t2 = PythonOperator(task_id='calculate_total_units', python_callable=calculate_total_units)
    t3 = PythonOperator(task_id='generate_bill_summary', python_callable=generate_bill_summary)
    t1 >> t2 >> t3

# Trigger in Colab
create_bill_file()
generate_bill_summary()

✔ Created /tmp/electricity.txt
Calculated: Customers=5, Total=1100, Avg=220.0
✔ Generated Summary at /tmp/bill_summary.txt

📋 File Content:
Customers = 5
Total Units = 1100
Average Units = 220

